<center>
<a href="http://www.insa-toulouse.fr/" ><img src="http://www.math.univ-toulouse.fr/~besse/Wikistat/Images/logo-insa.jpg" style="float:left; max-width: 120px; display: inline" alt="INSA"/></a> 

</center>

# Machine Learning Project
MoraGarcia Carmen, (i don't know your last name) Léo, Dubouchet Chloé et Perrin Alicia


## Introduction

The data is taken from the KAGGLE competition website; it is the data set "Cardivascular Disease Risk
Prediction Dataset" available here: https://www.kaggle.com/datasets/bertnardomariouskono/cardiovasculardisease-risk-prediction-dataset.


This dataset contains 15,000 synthetic patient medical records specifically designed to predict the risk of
cardiovascular disease. Although synthetic, the data is generated using medical heuristics to ensure realistic
correlations between variables, such as the relationship between age, BMI, and blood pressure. The dataset
includes 19 variables for 15,000 patients.

**WARNING** In order to have exactly the same training and test samples in both languages ​​so that their results can be compared afterwards, it is imperative to run the python notebook before this one.

In [2]:
# Chargement des librairies nécessaires
options(warn = -1)
library(ggplot2)
library(tidyverse)
library(gridExtra)
library(GGally)
library(plotly)
library(corrplot)
library(reshape2)
library(FactoMineR) 
library(factoextra)
library(glmnet) 
library(ggfortify)
library(pROC)
library(ROCR)

## 1. Exploratory data analysis

## a. Read the table of data

In [13]:
df<-read.csv("archive/healthcare_synthetic_data.csv", sep=",")
head(df)

,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,Systolic_BP,Diastolic_BP,Cholesterol_Total,Cholesterol_LDL,Cholesterol_HDL,Fasting_Blood_Sugar,Smoking_Status,Alcohol_Consumption,Physical_Activity_Level,Family_History,Stress_Level,Sleep_Hours,Heart_Disease_Risk
,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,PID-00001,60,0,146.9,51.3,23.8,140,89,217,151,52,83,0,1,3,0,1,8,0
2,PID-00002,53,0,161.8,76.6,29.3,128,81,203,119,38,116,0,0,1,0,7,9,0
3,PID-00003,62,1,174.7,92.4,30.3,141,100,173,124,45,90,0,0,0,0,1,7,1
4,PID-00004,73,1,173.3,68.9,22.9,136,96,193,117,45,81,0,0,1,0,2,7,1
5,PID-00005,52,1,178.6,79.8,25.0,122,80,236,153,41,79,0,1,2,0,2,6,0
6,PID-00006,52,0,159.6,60.3,23.7,134,92,225,155,48,103,0,0,1,1,4,8,0


We start by checking the nature of the different variables and their encoding.

In [14]:
df$Gender <- factor(df$Gender, labels = c("Woman", "Man"))

df$Smoking_Status <- factor(df$Smoking_Status, labels = c("Non_Smoker", "Smoker"))

df$Alcohol_Consumption <- factor(df$Alcohol_Consumption, labels = c("Non", "Moderate", "Heavy"))

df$Physical_Activity_Level <- factor(df$Physical_Activity_Level)

df$Family_History <- factor(df$Family_History, labels = c("No", "Yes"))

df$Heart_Disease_Risk <- factor(df$Heart_Disease_Risk, labels = c("Low", "High"))

Secondly, we transform some variable to be Gaussian.

In [15]:
df <- df %>%
  mutate(
    R_Weight_kg = sqrt(Weight_kg),
    R_BMI = sqrt(BMI)
  )

variables <- c('Age', 'Height_cm', 'R_Weight_kg', 'R_BMI', 'Systolic_BP', 
               'Diastolic_BP', 'Cholesterol_Total', 'Cholesterol_LDL', 
               'Cholesterol_HDL', 'Fasting_Blood_Sugar', 'Stress_Level', 
               'Sleep_Hours')

df <- df %>% 
  select(-BMI, -Weight_kg)

summary(df)

  Patient_ID             Age          Gender       Height_cm    
 Length:15000       Min.   :25.00   Woman:7622   Min.   :138.5  
 Class :character   1st Qu.:46.00   Man  :7378   1st Qu.:158.5  
 Mode  :character   Median :55.00                Median :164.7  
                    Mean   :54.54                Mean   :165.3  
                    3rd Qu.:63.00                3rd Qu.:172.0  
                    Max.   :85.00                Max.   :198.1  
  Systolic_BP     Diastolic_BP    Cholesterol_Total Cholesterol_LDL
 Min.   : 90.0   Min.   : 60.00   Min.   :127.0     Min.   : 70.0  
 1st Qu.:127.0   1st Qu.: 85.00   1st Qu.:201.0     1st Qu.:128.0  
 Median :135.0   Median : 91.00   Median :216.0     Median :140.0  
 Mean   :135.1   Mean   : 90.54   Mean   :216.2     Mean   :140.4  
 3rd Qu.:143.0   3rd Qu.: 96.00   3rd Qu.:231.0     3rd Qu.:152.0  
 Max.   :182.0   Max.   :120.00   Max.   :303.0     Max.   :210.0  
 Cholesterol_HDL Fasting_Blood_Sugar    Smoking_Status  Alcohol_Consu

# 2. Prediction of Heart’s Disease Risk
We consider the problem of predicting the variable Heart_Disease_Risk from the other variables from a machine learning point of view, i.e. focusing on model performance. The aim is to determine the best performance we can expect, and which models achieve it.

## a. Importation of training sample and a test sample

We decided to export the indices of the individuals in each sample so that we could compare the results obtained in R and in Python on the same data.

In [18]:
train_idx <- read.csv("train_indices.csv")$index
test_idx <- read.csv("test_indices.csv")$index

train_idx_r <- train_idx + 1
test_idx_r <- test_idx + 1

df_train <- df[train_idx_r, ]
df_test <- df[test_idx_r, ]